<a href="https://colab.research.google.com/github/Eswar2005-Karanam/blog/blob/main/spam_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import io
import re
import urllib.request
import zipfile
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==========================================
# 1. ROBUST DATA LOADING PIPELINE
# ==========================================
def load_spam_dataset():
    """Loads dataset directly from UCI repository with fallback synthetic data."""
    url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
    try:
        print("📥 Fetching UCI SMS Spam Collection dataset...")
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=10) as response:
            zip_file = zipfile.ZipFile(io.BytesIO(response.read()))
            with zip_file.open("SMSSpamCollection") as f:
                df = pd.read_csv(f, sep='\t', names=['label', 'text'])
        print(f" SUCCESS: Loaded {len(df)} records from official source.\n")
        return df
    except Exception as e:
        print(f"⚠️ Primary dataset download failed ({e}). Loading built-in fallback dataset...")
        # Emergency backup dataset to guarantee zero execution errors in Colab
        fallback_data = {
            "label": ["ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam"] * 20,
            "text": [
                "Hey, what time are we meeting today for the group project?",
                "WINNER! You have won a $1000 cash prize! Claim now at http://win.com",
                "Can you please send me the class lecture notes?",
                "URGENT: Your credit card account needs verification. Click here immediately.",
                "Let's get lunch near the campus canteen around 1 PM.",
                "Free ringtones for your phone! Text YES to 80085 to claim.",
                "Don't forget to submit the Python assignment before midnight.",
                "Congratulations! You are selected for a free holiday trip to Paris.",
                "Are you coming to the lab session today?",
                "Get cheap loans instantly with 0% interest rate. Call 1800-SPAM-NOW"
            ] * 20
        }
        return pd.DataFrame(fallback_data)

df = load_spam_dataset()

# ==========================================
# 2. TEXT CLEANING & PREPROCESSING
# ==========================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', ' url ', text) # Replace URLs with token
    text = re.sub(r'\b\d+\b', ' number ', text)              # Replace numbers
    text = re.sub(r'[^a-zA-r0-9\s]', '', text)              # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()                 # Remove extra whitespace
    return text

df['clean_text'] = df['text'].apply(clean_text)

# ==========================================
# 3. TRAIN / TEST SPLIT & VECTORIZATION
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# TF-IDF Vectorizer with unigrams + bigrams and stopword removal
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# ==========================================
# 4. MODEL TRAINING & COMPARISON
# ==========================================
# Baseline: Multinomial Naive Bayes
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)
nb_acc = accuracy_score(y_test, nb_preds)

# Improvement: Logistic Regression
lr_model = LogisticRegression(max_iter=1000, C=1.0)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, lr_preds)

# Output Summary Comparison Table
print("=" * 60)
print(f"{'MODEL PERFORMANCE COMPARISON':^60}")
print("=" * 60)
print(f"1. Multinomial Naive Bayes Accuracy : {nb_acc * 100:.2f}%")
print(f"2. Logistic Regression Accuracy    : {lr_acc * 100:.2f}%")
print("=" * 60)

print("\n📊 Detailed Classification Report (Logistic Regression):")
print(classification_report(y_test, lr_preds, target_names=['Ham (Legit)', 'Spam']))

# ==========================================
# 5. INFERENCE / CUSTOM TEXT TEST SUITE
# ==========================================
def predict_message(messages):
    print("\n" + "=" * 60)
    print(f"{'CUSTOM MESSAGE PREDICTIONS':^60}")
    print("=" * 60)

    cleaned_msgs = [clean_text(m) for m in messages]
    tfidf_msgs = vectorizer.transform(cleaned_msgs)

    preds = lr_model.predict(tfidf_msgs)
    probs = lr_model.predict_proba(tfidf_msgs)

    for original, pred, prob in zip(messages, preds, probs):
        confidence = max(prob) * 100
        tag = "🔴 [SPAM]" if pred == "spam" else "🟢 [HAM]"
        print(f"Input Text : \"{original}\"")
        print(f"Prediction : {tag} (Confidence: {confidence:.2f}%)")
        print("-" * 60)

# Sample test run
sample_inputs = [
    "URGENT! You have won $10,000 cash prize! Reply WIN to claim now.",
    "Hey Reddy, can you send over the updated project code before the meeting?",
    "Congratulations! You are selected for an exclusive gift coupon. Click bit.ly/claim-free",
    "Please review the attached PDF for today's lab instructions."
]

predict_message(sample_inputs)

📥 Fetching UCI SMS Spam Collection dataset...
 SUCCESS: Loaded 5572 records from official source.

                MODEL PERFORMANCE COMPARISON                
1. Multinomial Naive Bayes Accuracy : 98.21%
2. Logistic Regression Accuracy    : 97.40%

📊 Detailed Classification Report (Logistic Regression):
              precision    recall  f1-score   support

 Ham (Legit)       0.98      0.99      0.99       966
        Spam       0.95      0.85      0.90       149

    accuracy                           0.97      1115
   macro avg       0.96      0.92      0.94      1115
weighted avg       0.97      0.97      0.97      1115


                 CUSTOM MESSAGE PREDICTIONS                 
Input Text : "URGENT! You have won $10,000 cash prize! Reply WIN to claim now."
Prediction : 🔴 [SPAM] (Confidence: 98.27%)
------------------------------------------------------------
Input Text : "Hey Reddy, can you send over the updated project code before the meeting?"
Prediction : 🟢 [HAM] (Confidence